***Process Order Data***

In [0]:
create or refresh streaming table bronze_orders
comment "raw orders data ingested from the source system operational data"
tblproperties("quality"="bronze")
as
select *,
    _metadata.file_path as file_path,
    current_timestamp as ingestion_timestamp
from cloud_files(
    "/Volumes/circuitbox_sivan/landing/operational_data/orders/",
    "json",
    map('cloudFiles.inferColumnTypes', 'true')
)

Silver Orders Clean
1. Fail if customer_id is NULL
2. Fail if order_id is NULL
3. Warn if order_status is not one of pending, shipped, cancelled or completed.
4. Warn if payment_method is not one of Credit Card, Bank Transfer, Paypal

silver_orders_clean Transformation/
    1. Cast Order_timestamp to timestamp/
silver_orders to include all orders with the items array exploded. (i.e one record per each item in the order )

In [0]:
create or refresh streaming table silver_orders_clean(
    constraint valid_customer_id expect(customer_id is not null) on violation fail update,
    constraint valid_order_id expect(order_id is not null) on violation fail update,
    constraint valid_status expect(order_status in ('Pending','Shipped','Cancelled','Completed')),
    constraint valid_payment_method expect(payment_method in ('Credit Card','Bank Transfer','PayPal'))  
)
comment 'cleaned orders data'
tblproperties("quality"="silver")
as
select
    order_id,
    customer_id,
    CAST(order_timestamp as timestamp) as order_timestamp,
    payment_method,
    items,
    order_status
from stream(live.bronze_orders)



***3.Explode item array from the order object - silver_orders***

In [0]:
create or refresh streaming table silver_orders
comment "Expanded orders data"
tblproperties("quality"="silver")
as
select
    order_id,
    customer_id,
    order_timestamp,
    payment_method,
    item.item_id,
    item.name as item_name,
    item.category as item_category,
    item.price as item_price,
    item.quantity as item_quantity,
    order_status
from
(
    select
        order_id,
        customer_id,
        order_timestamp,
        payment_method,
        explode(items) as item,
        order_status
    from stream(live.silver_orders_clean)
)
